In [1]:
import os
import shutil

BASE_DIR = "/app"
DATA_PROCESSED = os.path.join(BASE_DIR, "data", "processed")

zip_path = os.path.join(DATA_PROCESSED, "valuation_ml_ready.parquet.zip")
unzip_target = DATA_PROCESSED

if os.path.exists(zip_path):
    print("Found zip:", zip_path)
    # Unpack zip contents into processed folder
    shutil.unpack_archive(zip_path, unzip_target)
    print("Unzipped into:", unzip_target)
else:
    print("No zip found, continuing (expected if parquet folder already exists).")

No zip found, continuing (expected if parquet folder already exists).


In [2]:
import os

BASE_DIR = "/app"
DATA_PROCESSED = os.path.join(BASE_DIR, "data", "processed")
OUTPUTS_DIR = os.path.join(BASE_DIR, "outputs")

INPUT_PATH = os.path.join(DATA_PROCESSED, "valuation_ml_ready.parquet")
OUTPUT_PATH = os.path.join(DATA_PROCESSED, "valuation_ml_cv_ready.parquet")

# sanity check
if not os.path.exists(INPUT_PATH):
    raise FileNotFoundError(
        f"Input parquet not found at: {INPUT_PATH}\n"
        "Run 02_feature_engineering.ipynb first to generate valuation_ml_ready.parquet."
    )

In [3]:
# Using Spark
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.appName("valuation-folds").getOrCreate()

df = spark.read.parquet(INPUT_PATH)

print("Rows:", df.count())
df.select("row_id").distinct().count() == df.count()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/12 19:57:45 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/02/12 19:57:46 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/02/12 19:57:46 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
                                                                                

Rows: 88092


True

In [4]:
# Assign deterministic fold_id (Should be 0-9)
df_folds = df.withColumn("fold_id", (F.col("row_id") % F.lit(10)).cast("int"))

# Fold balance check
df_folds.groupBy("fold_id").count().orderBy("fold_id").show(20, truncate=False)

# Label distribution per fold
df_folds.groupBy("fold_id", "label").count().orderBy("fold_id", "label").show(200, truncate=False)

+-------+-----+
|fold_id|count|
+-------+-----+
|0      |8809 |
|1      |8810 |
|2      |8810 |
|3      |8809 |
|4      |8809 |
|5      |8809 |
|6      |8809 |
|7      |8809 |
|8      |8809 |
|9      |8809 |
+-------+-----+

+-------+-----+-----+
|fold_id|label|count|
+-------+-----+-----+
|0      |0.0  |2952 |
|0      |1.0  |2897 |
|0      |2.0  |2960 |
|1      |0.0  |2959 |
|1      |1.0  |2962 |
|1      |2.0  |2889 |
|2      |0.0  |2904 |
|2      |1.0  |2930 |
|2      |2.0  |2976 |
|3      |0.0  |2981 |
|3      |1.0  |2956 |
|3      |2.0  |2872 |
|4      |0.0  |2929 |
|4      |1.0  |2887 |
|4      |2.0  |2993 |
|5      |0.0  |2984 |
|5      |1.0  |2878 |
|5      |2.0  |2947 |
|6      |0.0  |2932 |
|6      |1.0  |2992 |
|6      |2.0  |2885 |
|7      |0.0  |2897 |
|7      |1.0  |2930 |
|7      |2.0  |2982 |
|8      |0.0  |2924 |
|8      |1.0  |2931 |
|8      |2.0  |2954 |
|9      |0.0  |2870 |
|9      |1.0  |2961 |
|9      |2.0  |2978 |
+-------+-----+-----+



In [5]:
#Output
df_folds.write.mode("overwrite").parquet(OUTPUT_PATH)

print("Success: ", OUTPUT_PATH)

Success:  /app/data/processed/valuation_ml_cv_ready.parquet


In [6]:
import os
import shutil

artifacts_dir = os.path.join(OUTPUTS_DIR, "artifacts")
os.makedirs(artifacts_dir, exist_ok=True)

zip_path = os.path.join(artifacts_dir, "valuation_ml_cv_ready.zip")
base_name = zip_path.replace(".zip", "")

# Zip the parquet output folder for sharing/submission if needed
shutil.make_archive(base_name, "zip", OUTPUT_PATH)

print("Created:", zip_path)

Created: /app/outputs/artifacts/valuation_ml_cv_ready.zip
